In [1]:
import pandas as pd
df = pd.read_csv("messy_sales.csv")

In [2]:
df.shape

(300, 6)

In [3]:
df.head(10)

,order_id,date,product,price,qty,zip
0,1254,04/06/2026,mug,11.36,4,60614
1,1057,2026-05-30,charger,14.79,3,30303
2,1150,05/06/2026,mug,5.50,1,10001
3,1066,05/30/2026,notebook,NaN,1,98101
4,1114,04/06/2026,webcam,45.59,4,90405
5,1280,04/22/2026,pen set,NaN,4,30303
6,1204,04/18/2026,charger,8.41,2,2134
7,1249,2026-05-15,keyboard,75.04,4,90405
8,1037,2026-05-19,mug,9.31,2,60614
9,1177,2026-05-13,mug,10.11,3,90210


In [4]:
df.dtypes

order_id      int64
date            str
product         str
price       float64
qty           int64
zip           int64
dtype: object

I can already see that the orders with id 1066 and 1280 are missing their prices. I see that the format of the date is not constant among the items. The type of date is str which would be better with datetime64.

In [5]:
df.isna().sum()

order_id     0
date         0
product      0
price       12
qty          0
zip          0
dtype: int64

In [6]:
fill_value = df["price"].median()
df["price"] = df["price"].fillna(fill_value)

In [7]:
df.isna().sum()

order_id    0
date        0
product     0
price       0
qty         0
zip         0
dtype: int64

In [8]:
print(fill_value)

37.53


In [9]:
print(df.duplicated().sum())

8


In [10]:
df = df.drop_duplicates()
df.shape

(292, 6)

In [11]:
df["zip"] = df["zip"].astype(str).str.zfill(5)

In [12]:
df

,order_id,date,product,price,qty,zip
0,1254,04/06/2026,mug,11.36,4,60614
1,1057,2026-05-30,charger,14.79,3,30303
2,1150,05/06/2026,mug,5.50,1,10001
3,1066,05/30/2026,notebook,37.53,1,98101
4,1114,04/06/2026,webcam,45.59,4,90405
...,...,...,...,...,...,...
294,1103,2026-04-21,pen set,16.47,3,90405
296,1067,2026-04-27,backpack,37.93,5,90210
297,1025,2026-04-27,keyboard,47.30,-2,02116
298,1196,04/25/2026,pen set,8.79,3,90405


In [13]:
df["date"] = pd.to_datetime(df["date"], format="mixed")

In [14]:
df

,order_id,date,product,price,qty,zip
0,1254,2026-04-06,mug,11.36,4,60614
1,1057,2026-05-30,charger,14.79,3,30303
2,1150,2026-05-06,mug,5.50,1,10001
3,1066,2026-05-30,notebook,37.53,1,98101
4,1114,2026-04-06,webcam,45.59,4,90405
...,...,...,...,...,...,...
294,1103,2026-04-21,pen set,16.47,3,90405
296,1067,2026-04-27,backpack,37.93,5,90210
297,1025,2026-04-27,keyboard,47.30,-2,02116
298,1196,2026-04-25,pen set,8.79,3,90405


In [15]:
df.dtypes

order_id             int64
date        datetime64[us]
product                str
price              float64
qty                  int64
zip                    str
dtype: object

In [16]:
df.shape

(292, 6)

In [17]:
df[df["qty"] < 0]

,order_id,date,product,price,qty,zip
202,1140,2026-04-18,webcam,38.69,-5,98101
262,1233,2026-05-09,desk lamp,22.64,-4,10001
297,1025,2026-04-27,keyboard,47.30,-2,02116


In [18]:
df.describe()

,order_id,date,price,qty
count,292.000000,292,292.000000,292.000000
mean,1145.500000,2026-05-01 16:50:57.534246,55.523151,2.910959
min,1000.000000,2026-04-01 00:00:00,2.930000,-5.000000
25%,1072.750000,2026-04-18 00:00:00,11.045000,2.000000
50%,1145.500000,2026-05-02 00:00:00,37.530000,3.000000
75%,1218.250000,2026-05-15 00:00:00,55.510000,4.000000
max,1291.000000,2026-05-30 00:00:00,479.290000,5.000000
std,84.437354,NaN,71.767085,1.559457


In [19]:
df["qty"] = df["qty"].abs()

In [20]:
df[df["qty"] < 0]

,order_id,date,product,price,qty,zip


In [21]:
df.to_csv("sales_clean.csv", index=False)

# Cleaning Log
300 loaded; 12 prices filled with 37.53; decided to use median value to fill instead of mean, since the data is skewed; 8 duplicates removed leaving 292; zips restored; dates standardized; negatives handled by wrapping them with absolute value to include them in the data. I decided to include them because there were only 3 of them out of 292, and the prices of all of them were pretty low and I thought they wouldn't make a significant change to the data and the overall analysis. In actual business, I could see how you could decide to ignore or include a single data, which could have a significant impact on the data analysis. Using that wrong analysis, the business could go downfall in the blink of an eye. For example, if one of the items had -5000 qty that are worth millions of dollars, the decision to include that negative quantity could lead to some big trouble for the business.

Repo Address: https://github.com/yeonghwang-hub/cs82a-portfolio